# Experimento de 29/10/2026 — versão organizada para reprodução

Este notebook organiza o fluxo do notebook atualizado fornecido pelo autor, preservando os parâmetros centrais de **Quickshift → K-Means (6 clusters) → NDVI/brilho HSV/bordas de Canny → Random Forest (500 estimadores) → estimativa alométrica C3 → proximidade de postes (1, 3 e 5 m)**.

**Resultados registrados no notebook original, não recalculados aqui:** 458 pares objeto–rótulo; 366 treino; 92 teste; acurácia arredondada de 87%.

**Ajuste em relação ao código recebido:** a etapa espacial usa apenas previsões da classe `arvore`, evitando calcular altura para previsões de `grama`. Por isso, as saídas espaciais devem ser reexecutadas e conferidas.

**Atenção:** o lote `Area1` traz `postes.gpkg`, mas não a geometria das linhas de energia. A configuração da altura `2.5 × área**0.4` é experimental e ainda não validada com medições de campo.

É recomendado usar o **Google Colab com RAM alta**. Os rasters abrangem cerca de 50,6 milhões de pixels e a execução integral pode ser pesada.

In [ ]:
# 0 — Ambiente

# Execute esta célula uma vez no Google Colab.
%pip -q install rasterio geopandas scikit-image scikit-learn pandas matplotlib seaborn shapely
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from rasterio.mask import mask
from shapely.geometry import shape
from skimage.segmentation import quickshift, mark_boundaries
from skimage.feature import canny
from skimage.color import rgb2hsv
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive
drive.mount('/content/drive')

# Extraia o ZIP enviado dentro da pasta abaixo (a pasta Area1 tem os 10 arquivos).
DATA_DIR = Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/Area1')
OUT_DIR = Path('/content/drive/MyDrive/Projetos/Segmentacao_Breno/outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
necessarios = ['AOI_RGB_NEW.tif','AOI_NEW_MULTI.tif','AOI_new.shp','Arvores.gpkg','Grama.gpkg','postes.gpkg']
ausentes = [nome for nome in necessarios if not (DATA_DIR / nome).exists()]
assert not ausentes, f'Arquivos ausentes em {DATA_DIR}: {ausentes}'
print('Dados encontrados em', DATA_DIR)

In [ ]:
# 1–3 — Rasters, segmentação e objetos georreferenciados

with rasterio.open(DATA_DIR/'AOI_RGB_NEW.tif') as rgb:
    im = rgb.read().transpose(1, 2, 0)
    transform_rgb, crs_rgb = rgb.transform, rgb.crs
AOI = gpd.read_file(DATA_DIR/'AOI_new.shp').to_crs(crs_rgb)
assert crs_rgb.is_projected, 'É necessário CRS projetado para calcular área/distância em metros'
print('Imagem RGB:', im.shape, 'CRS:', crs_rgb)

# Parâmetros conforme o notebook fornecido; cálculo pesado para toda a área!
segments_quick2 = quickshift(im, kernel_size=3, max_dist=5, ratio=0.04).astype('uint32')
print('Total de rótulos de segmento:', np.unique(segments_quick2).size)

# Converter os segmentos em polígonos, mantendo buracos de geometria, quando existirem.
polygons = (shape(geom) for geom, value in features.shapes(segments_quick2, transform=transform_rgb))
Poly_gdf = gpd.GeoDataFrame(geometry=list(polygons), crs=crs_rgb)
clip_poly = Poly_gdf.clip(AOI).reset_index(drop=True)
print('Polígonos na área de interesse:', len(clip_poly))

In [ ]:
# 4–6 — Características RGB–NIR e K-Means

with rasterio.open(DATA_DIR/'AOI_NEW_MULTI.tif') as src_mult:
    assert src_mult.count >= 4, 'A imagem deve conter R, G, B e NIR'
    def stats_por_objeto(geom):
        bloco, _ = mask(src_mult, [geom], crop=True, nodata=0)
        medias, desvios = [], []
        for banda in bloco[:4]:
            valido = banda[banda != 0]
            medias.append(float(valido.mean()) if valido.size else np.nan)
            desvios.append(float(valido.std()) if valido.size else np.nan)
        return medias + desvios

    registros = [stats_por_objeto(g.__geo_interface__) for g in clip_poly.geometry]

colunas = ['R_Mean','G_Mean','B_Mean','N_Mean','R_Std','G_Std','B_Std','N_Std']
tabela = pd.DataFrame(registros, columns=colunas)
validos = tabela.dropna().index
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
clip_poly['kmeans_cluster'] = np.nan
clip_poly.loc[validos, 'kmeans_cluster'] = kmeans.fit_predict(tabela.loc[validos, colunas])
print('Objetos por cluster:', clip_poly['kmeans_cluster'].value_counts().to_dict())

# O rótulo numérico não tem interpretação física fixa; aqui 3 reproduz a escolha feita no notebook original.
CLUSTER_VEGETACAO = 3
candidatos = clip_poly.loc[clip_poly.kmeans_cluster == CLUSTER_VEGETACAO].copy()
print('Candidatos de vegetação:', len(candidatos))

In [ ]:
# 7–10 — Rótulos, NDVI, brilho e bordas

grama = gpd.read_file(DATA_DIR/'Grama.gpkg').to_crs(crs_rgb).copy()
arvores = gpd.read_file(DATA_DIR/'Arvores.gpkg').to_crs(crs_rgb).copy()
grama['Classe'], arvores['Classe'] = 'grama', 'arvore'
rotulos = gpd.GeoDataFrame(pd.concat([grama, arvores], ignore_index=True), crs=crs_rgb)
amostras = gpd.sjoin(candidatos, rotulos[['Classe','geometry']], how='inner', predicate='intersects')
print('Pares objeto–rótulo:', len(amostras), 'objetos distintos:', amostras.index.nunique())

with rasterio.open(DATA_DIR/'AOI_NEW_MULTI.tif') as src_multi:
    raster = src_multi.read()
    metadata = src_multi.profile
    mult_transform = src_multi.transform
r, g, b, nir = [raster[i] for i in range(4)]
ndvi = (nir.astype('float32')-r.astype('float32')) / (nir.astype('float32')+r.astype('float32')+1e-8)
brilho = rgb2hsv(np.stack([r,g,b],axis=-1))[:,:,2]
bordas = canny(nir / 10000.0, sigma=1)
del raster
stack = np.stack([brilho.astype('float32'), bordas.astype('float32'), ndvi.astype('float32')])
metadata.update(dtype='float32', count=3, nodata=None)
features_path = OUT_DIR/'features_stack.tif'
with rasterio.open(features_path, 'w', **metadata) as dst:
    dst.write(stack)
del stack, brilho, bordas, ndvi
print('Características gravadas em', features_path)

In [ ]:
# 11–12 — Random Forest, matriz de confusão e classificação

def extrair_features(gdf, caminho):
    linhas = []
    with rasterio.open(caminho) as src:
        for geom in gdf.geometry:
            bloco, _ = mask(src, [geom.__geo_interface__], crop=True, nodata=-999)
            medias = []
            for banda in bloco:
                valido = banda[banda != -999]
                medias.append(float(valido.mean()) if valido.size else np.nan)
            linhas.append(medias)
    return pd.DataFrame(linhas, index=gdf.index, columns=['mean_intensity','mean_edges','mean_ndvi'])

atributos = extrair_features(amostras, features_path)
atributos['classe'] = amostras['Classe'].to_numpy()
atributos = atributos.dropna()
X, y = atributos.drop(columns='classe'), atributos['classe']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_classifier = RandomForestClassifier(n_estimators=500, random_state=42)
rf_classifier.fit(X_train, y_train)
pred_teste = rf_classifier.predict(X_test)
print('Amostras:', len(X), 'Treino:', len(X_train), 'Teste:', len(X_test))
print(classification_report(y_test, pred_teste))
print('Acurácia:', accuracy_score(y_test, pred_teste))
ConfusionMatrixDisplay.from_predictions(y_test, pred_teste)
plt.show()

# Prever rótulos apenas para candidatos com feições válidas.
atrib_pred = extrair_features(candidatos, features_path).dropna()
candidatos['predicted_classe'] = pd.Series(dtype='object')
candidatos.loc[atrib_pred.index, 'predicted_classe'] = rf_classifier.predict(atrib_pred)
print(candidatos['predicted_classe'].value_counts(dropna=False))

In [ ]:
# 13–15 — Copas e altura empírica (C3)

# Correção frente ao notebook recebido: excluir objetos classificados como gramínea.
tree_polys = candidatos.loc[candidatos['predicted_classe'] == 'arvore'].copy()
tree_polys['area_m2'] = tree_polys.geometry.area
# Adaptação empírica; requer validação contra alturas reais.
tree_polys['altura_estimada_m'] = 2.5 * (tree_polys['area_m2'] ** 0.4)
filter_tree = tree_polys.loc[tree_polys.altura_estimada_m > 7].copy()
print('Previstas como árvores:', len(tree_polys), 'com altura estimada > 7m:', len(filter_tree))
filter_tree.to_file(OUT_DIR/'arvores_estimadas_maiores_7m.gpkg', driver='GPKG')

In [ ]:
# 16–17 — Proximidade de postes (1 m, 3 m, 5 m)

postes = gpd.read_file(DATA_DIR/'postes.gpkg').to_crs(crs_rgb).clip(AOI)
print('Postes na área:', len(postes), 'árvores candidatas:', len(filter_tree))
for distancia in (1, 3, 5):
    vizinhos = gpd.sjoin_nearest(filter_tree, postes, how='inner',
                                 max_distance=distancia, distance_col='distancia_m')
    # Cada objeto pode aparecer mais de uma vez, se houver múltiplos postes.
    total_arvores = vizinhos.index.nunique()
    print(f'Até {distancia}m: {total_arvores} árvores únicas; {len(vizinhos)} pares árvore–poste')
    vizinhos.to_file(OUT_DIR/f'arvores_proximas_postes_{distancia}m.gpkg', driver='GPKG')
print('Concluído: resultados espaciais em', OUT_DIR)

As análises de proximidade são **bidimensionais** e dizem respeito a postes, não aos condutores. Não significam, por si só, risco elétrico confirmado.

Este notebook é uma implementação organizada e reprodutível baseada no código recebido, com uma correção explícita no filtro de árvores. **Os resultados registrados no arquivo recebido (acurácia 0,87) não foram reproduzidos automaticamente por esta atualização.** Confira as métricas após executar este notebook.

Próximos experimentos: divisão de treino/teste espacialmente independente, deduplicação de objetos com rótulos sobrepostos, comparação com outras técnicas, ablação das três características e validação da altura com medidas reais.

Veja também `docs/experimento_area_completa_2026-10-29.md` para resultados, ressalvas científicas e dados de entrada.